# A7：Context Management — 四種策略實作與 Token 監控

> **對應 Blog：** FDE 面試準備指南（十）AI Agent 的 Context Management  
> **核心目標：** 親手看到 context 在第 50 輪時如何爆炸，再親手實作四種策略讓它不爆。

## 面試情境（必考題）

> 「你的 multi-turn Agent 在第 50 輪對話時，會發生什麼問題？你怎麼設計解決它？」

## 學習路徑

| Part | 內容 | 關鍵觀察 |
|------|------|----------|
| 1 | Token 增長模擬 | 親眼看到第 50 輪快爆了 |
| 2 | 策略一：Sliding Window | 最簡單，但會丟訊息 |
| 3 | 策略二：Summary Buffer | 壓縮保語意，但有額外成本 |
| 4 | 策略三：Retrieval Memory | 向量召回，最彈性 |
| 5 | 策略四：Structured State | Context 大小永遠固定 |
| 6 | Token 監控 + 自動觸發 | 生產環境的護欄 |
| 7 | Lost-in-the-Middle 實驗 | 為什麼順序很重要 |
| 8 | 面試 SCOPE 框架 | 白板語言 |

In [ ]:
import os
import asyncio
import json
import time
from typing import TypedDict
from collections import deque
from dotenv import load_dotenv

load_dotenv()

try:
    import tiktoken
    enc = tiktoken.encoding_for_model("gpt-4o")
    def count_tokens(text: str) -> int:
        return len(enc.encode(text))
    print("✅ tiktoken 可用（精確 token 計數）")
except ImportError:
    def count_tokens(text: str) -> int:
        return len(text.split()) * 4 // 3  # 粗估：英文約 0.75 詞/token
    print("⚠️  tiktoken 未安裝，使用近似計數（pip install tiktoken）")

try:
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    LLM_AVAILABLE = True
    print("✅ LLM 可用")
except:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定，使用模擬輸出")

MAX_CONTEXT_TOKENS = 128_000  # GPT-4o 的 context window
WARNING_THRESHOLD = 0.60      # 60% 開始警告
FORCE_COMPRESS_THRESHOLD = 0.80  # 80% 強制壓縮

print(f"\nContext Window: {MAX_CONTEXT_TOKENS:,} tokens")
print(f"Warning at: {WARNING_THRESHOLD*100:.0f}% ({int(MAX_CONTEXT_TOKENS*WARNING_THRESHOLD):,} tokens)")
print(f"Force compress at: {FORCE_COMPRESS_THRESHOLD*100:.0f}% ({int(MAX_CONTEXT_TOKENS*FORCE_COMPRESS_THRESHOLD):,} tokens)")

## Part 1：親眼看到 Context 爆炸

In [ ]:
# 模擬典型客服 Agent 的 context 增長

SYSTEM_PROMPT = """你是一個專業的技術客服 AI，
負責協助用戶解決軟體問題，回答產品相關問題。
請用繁體中文回答，保持專業且友善的語氣。"""

# 模擬每輪對話的平均 token 數
AVG_USER_TOKENS = 80    # 用戶訊息
AVG_AGENT_TOKENS = 200  # AI 回應
AVG_TOKENS_PER_TURN = AVG_USER_TOKENS + AVG_AGENT_TOKENS

# 固定開銷
SYSTEM_TOKENS = count_tokens(SYSTEM_PROMPT)
RAG_CONTEXT_TOKENS = 1500  # RAG 召回的文件
OUTPUT_RESERVE = 2000       # 預留給輸出的空間
FIXED_OVERHEAD = SYSTEM_TOKENS + RAG_CONTEXT_TOKENS + OUTPUT_RESERVE

print("=" * 55)
print("Context 增長模擬（典型客服 Agent）")
print("=" * 55)
print(f"{'輪次':>6} | {'歷史 tokens':>12} | {'總計':>10} | {'使用率':>8} | 狀態")
print("-" * 55)

checkpoints = [1, 5, 10, 20, 50, 100, 200]
for turn in checkpoints:
    history_tokens = turn * AVG_TOKENS_PER_TURN
    total = FIXED_OVERHEAD + history_tokens
    usage = total / MAX_CONTEXT_TOKENS
    
    if usage > 1.0:
        status = "💥 OVERFLOW"
    elif usage > FORCE_COMPRESS_THRESHOLD:
        status = "🔴 強制壓縮"
    elif usage > WARNING_THRESHOLD:
        status = "🟡 警告"
    else:
        status = "🟢 正常"
    
    print(f"{turn:>6} | {history_tokens:>12,} | {total:>10,} | {usage:>7.1%} | {status}")

print()
print(f"固定開銷：{FIXED_OVERHEAD:,} tokens")
print(f"每輪新增：{AVG_TOKENS_PER_TURN} tokens")
safe_turns = (MAX_CONTEXT_TOKENS * WARNING_THRESHOLD - FIXED_OVERHEAD) // AVG_TOKENS_PER_TURN
print(f"\n⚠️  在第 {safe_turns:.0f} 輪時觸發警告 — 比你想像的早！")

## Part 2：策略一 — Sliding Window（最簡單）

In [ ]:
class SlidingWindowMemory:
    """
    Sliding Window：只保留最近 N 輪對話
    優點：實作最簡單，context 大小完全可預測
    缺點：超出視窗的資訊永久丟失
    """
    
    def __init__(self, max_turns: int = 10):
        self.max_turns = max_turns
        # deque 自動丟棄舊訊息
        self.history: deque[dict] = deque(maxlen=max_turns * 2)  # *2 因為每輪有 user+assistant
        self.total_turns = 0
        self.dropped_turns = 0
    
    def add_turn(self, user_msg: str, assistant_msg: str):
        self.total_turns += 1
        if len(self.history) >= self.history.maxlen:
            self.dropped_turns += 1
        self.history.append({"role": "user", "content": user_msg})
        self.history.append({"role": "assistant", "content": assistant_msg})
    
    def get_context(self) -> list[dict]:
        return list(self.history)
    
    def token_count(self) -> int:
        total = 0
        for msg in self.history:
            total += count_tokens(msg["content"])
        return total
    
    def stats(self) -> dict:
        return {
            "total_turns": self.total_turns,
            "kept_turns": len(self.history) // 2,
            "dropped_turns": self.dropped_turns,
            "token_count": self.token_count()
        }

# 測試：模擬 30 輪對話
sw = SlidingWindowMemory(max_turns=5)

conversations = [
    ("我的訂單 #1234 在哪裡？", "您的訂單 #1234 目前在配送中，預計明天到達。"),
    ("我想退換貨", "請問是什麼原因需要退換貨？"),
    ("產品有瑕疵", "非常抱歉！請您拍照後上傳，我們會儘速處理。"),
    ("怎麼更換密碼？", "請到設定頁面，點選帳戶安全，即可更換密碼。"),
    ("我忘記帳號了", "請提供您的手機號碼，我協助您找回帳號。"),
    ("之前那個訂單 #1234 退款了嗎？", "讓我查詢一下..."),  # 這輪詢問第 1 輪的資訊
]

print("=" * 55)
print("Sliding Window 測試（max_turns=5）")
print("=" * 55)

for i, (user, assistant) in enumerate(conversations, 1):
    sw.add_turn(user, assistant)
    s = sw.stats()
    print(f"第 {i:2d} 輪 | 保留 {s['kept_turns']} 輪 | 丟棄 {s['dropped_turns']} 輪 | "
          f"{s['token_count']} tokens")

print(f"\n⚠️  第 6 輪詢問訂單 #1234，但第 1 輪已被丟棄！")
print(f"   Agent 看不到 '訂單 #1234 在配送中' 的資訊")
print(f"   這就是 Sliding Window 的根本缺點")

print(f"\n✅ 但優點：token 數永遠固定在 ~{sw.stats()['token_count']} tokens")

## Part 3：策略二 — Summary Buffer（壓縮保語意）

In [ ]:
class SummaryBufferMemory:
    """
    Summary Buffer：舊對話壓縮成摘要 + 保留近期完整歷史
    優點：保留語意連貫性
    缺點：壓縮需要額外 LLM 呼叫（成本 + 延遲）
    """
    
    def __init__(self, recent_turns: int = 5, max_summary_tokens: int = 500):
        self.recent_turns = recent_turns
        self.max_summary_tokens = max_summary_tokens
        self.summary: str = ""           # 壓縮後的歷史摘要
        self.recent: list[dict] = []     # 最近 N 輪完整對話
        self.all_history: list[dict] = [] # 完整歷史（用於壓縮）
        self.compression_count = 0
    
    def add_turn(self, user_msg: str, assistant_msg: str):
        self.all_history.append({"role": "user", "content": user_msg})
        self.all_history.append({"role": "assistant", "content": assistant_msg})
        self.recent.append({"role": "user", "content": user_msg})
        self.recent.append({"role": "assistant", "content": assistant_msg})
        
        # 當 recent 超過限制時，壓縮最舊的部分
        if len(self.recent) > self.recent_turns * 2:
            self._compress_oldest()
    
    def _compress_oldest(self):
        """把 recent 中最舊的對話壓縮成摘要"""
        # 取出要壓縮的對話（recent 中最舊的一半）
        to_compress = self.recent[:self.recent_turns]
        self.recent = self.recent[self.recent_turns:]
        self.compression_count += 1
        
        # 在生產環境：呼叫 LLM 壓縮
        # 這裡模擬壓縮結果
        compressed_text = " | ".join(
            f"[{m['role']}]: {m['content'][:50]}"
            for m in to_compress
        )
        
        if self.summary:
            self.summary = f"{self.summary}\n---\n{compressed_text}"
        else:
            self.summary = compressed_text
        
        print(f"  🗜️  [壓縮 #{self.compression_count}] {len(to_compress)//2} 輪 → 摘要")
    
    def get_context(self) -> dict:
        """回傳注入 LLM 的完整 context"""
        return {
            "summary": self.summary,
            "recent_history": self.recent,
            "total_tokens": self.token_count()
        }
    
    def token_count(self) -> int:
        summary_tokens = count_tokens(self.summary) if self.summary else 0
        recent_tokens = sum(count_tokens(m["content"]) for m in self.recent)
        return summary_tokens + recent_tokens


# 測試
sb = SummaryBufferMemory(recent_turns=3)

print("=" * 55)
print("Summary Buffer 測試（recent_turns=3）")
print("=" * 55)

for i, (user, assistant) in enumerate(conversations, 1):
    sb.add_turn(user, assistant)
    ctx = sb.get_context()
    print(f"第 {i:2d} 輪 | recent={len(ctx['recent_history'])//2} 輪 | "
          f"summary={len(sb.summary)} 字 | {ctx['total_tokens']} tokens")

print(f"\n📋 最終 Context 結構：")
ctx = sb.get_context()
print(f"  摘要（壓縮的舊對話）: {count_tokens(ctx['summary'])} tokens")
print(f"  近期完整對話: {sum(count_tokens(m['content']) for m in ctx['recent_history'])} tokens")
print(f"\n✅ 第 6 輪詢問訂單 #1234，信息存在摘要裡！（不會完全丟失）")

## Part 4：策略三 — Retrieval Memory（向量召回）

In [ ]:
import math

class SimpleRetrievalMemory:
    """
    Retrieval Memory：對話存向量 DB，每輪按相關性召回
    優點：能「想起來」幾個月前說過的話
    缺點：需要向量基礎設施 + 召回延遲
    
    這裡用 TF-IDF 近似模擬向量相似度（生產用 Vertex AI Embedding）
    """
    
    def __init__(self, top_k: int = 3):
        self.top_k = top_k
        self.memories: list[dict] = []  # 模擬向量 DB
    
    def add_memory(self, turn_id: int, user_msg: str, assistant_msg: str, 
                   importance: float = 0.5):
        """儲存對話到記憶（非同步，不在請求路徑上）"""
        self.memories.append({
            "turn_id": turn_id,
            "user": user_msg,
            "assistant": assistant_msg,
            "importance": importance,
            "text": f"{user_msg} {assistant_msg}"  # 用於相似度計算
        })
    
    def _simple_similarity(self, query: str, text: str) -> float:
        """簡單的詞集合相似度（模擬 embedding cosine similarity）"""
        q_words = set(query.lower().split())
        t_words = set(text.lower().split())
        if not q_words or not t_words:
            return 0.0
        intersection = q_words & t_words
        # Jaccard similarity
        return len(intersection) / len(q_words | t_words)
    
    def recall(self, query: str) -> list[dict]:
        """根據當前 query 召回最相關的記憶"""
        scored = []
        for mem in self.memories:
            sim = self._simple_similarity(query, mem["text"])
            # 重要性加權
            score = sim * 0.7 + mem["importance"] * 0.3
            scored.append((score, mem))
        
        scored.sort(key=lambda x: x[0], reverse=True)
        return [mem for score, mem in scored[:self.top_k] if score > 0.1]


# 測試
rm = SimpleRetrievalMemory(top_k=2)

print("=" * 55)
print("Retrieval Memory 測試")
print("=" * 55)

# 儲存對話歷史（模擬非同步寫入）
for i, (user, assistant) in enumerate(conversations, 1):
    # 重要性：含訂單號、問題描述的對話重要性更高
    importance = 0.8 if "#" in user or "退" in user else 0.4
    rm.add_memory(i, user, assistant, importance)

# 第 6 輪：用戶詢問訂單 #1234
query = "之前那個訂單 #1234 退款了嗎？"
recalled = rm.recall(query)

print(f"當前 Query: '{query}'")
print(f"\n召回的相關記憶（top {rm.top_k}）:")
for mem in recalled:
    print(f"  第 {mem['turn_id']} 輪（重要性={mem['importance']}）:")
    print(f"    Q: {mem['user']}")
    print(f"    A: {mem['assistant']}")

print(f"\n✅ 即使是第 1 輪的資訊，只要語意相關就能召回！")
print(f"   但注意：召回延遲 ~50-200ms（需要 embedding + similarity search）")

## Part 5：策略四 — Structured State（Context 大小永遠固定）

In [ ]:
from dataclasses import dataclass, field, asdict

@dataclass
class CustomerServiceState:
    """
    Structured State：不存對話，只維護結構化狀態
    Context 大小永遠固定，不隨輪次增長
    適合：有明確工作流的 task-oriented Agent
    """
    # 用戶身份
    user_id: str = ""
    user_name: str = ""
    
    # 當前任務狀態
    task_type: str = ""          # 退換貨 | 查詢訂單 | 帳號問題
    task_status: str = "open"    # open | pending | resolved
    
    # 關鍵數據（不用說「之前你說過」）
    order_id: str = ""
    issue_type: str = ""         # 瑕疵 | 尺寸不符 | 延遲
    photo_uploaded: bool = False
    
    # 對話進度追蹤
    steps_completed: list[str] = field(default_factory=list)
    pending_action: str = ""     # 等待用戶做什麼
    
    # Meta
    turn_count: int = 0
    
    def to_prompt_str(self) -> str:
        """轉換成注入 LLM 的 context 字串"""
        return f"""當前服務狀態：
  用戶：{self.user_name or '未知'}
  任務類型：{self.task_type or '未確定'}
  任務狀態：{self.task_status}
  訂單編號：{self.order_id or '未提供'}
  問題類型：{self.issue_type or '未確定'}
  照片已上傳：{'是' if self.photo_uploaded else '否'}
  等待用戶：{self.pending_action or '無'}
  已完成步驟：{', '.join(self.steps_completed) or '無'}"""
    
    def token_count(self) -> int:
        return count_tokens(self.to_prompt_str())


# 模擬 30 輪對話後的狀態
state = CustomerServiceState(
    user_id="u_123",
    user_name="王大明"
)

print("=" * 55)
print("Structured State 測試")
print("=" * 55)

# 模擬對話過程中狀態更新
state.turn_count = 1
print(f"第 1 輪後 context: {state.token_count()} tokens")

state.task_type = "退換貨"
state.order_id = "#1234"
state.turn_count = 3
print(f"第 3 輪後 context: {state.token_count()} tokens")

state.issue_type = "產品瑕疵"
state.steps_completed = ["確認訂單", "確認問題類型"]
state.pending_action = "等待用戶上傳照片"
state.turn_count = 10
print(f"第 10 輪後 context: {state.token_count()} tokens")

state.photo_uploaded = True
state.steps_completed.append("收到照片")
state.pending_action = "等待倉庫確認"
state.turn_count = 50  # 第 50 輪
print(f"第 50 輪後 context: {state.token_count()} tokens")

print(f"\n✅ 無論第幾輪，context 大小基本不變！")
print(f"   對比 Sliding Window 第 10 輪：~{10 * AVG_TOKENS_PER_TURN} tokens")

print(f"\n📋 第 50 輪的完整 Context：")
print(state.to_prompt_str())

## Part 6：Token 監控 + 自動觸發壓縮

In [ ]:
class TokenMonitor:
    """生產環境的 Token 監控與自動壓縮觸發"""
    
    def __init__(self, max_tokens: int = MAX_CONTEXT_TOKENS,
                 warning_threshold: float = WARNING_THRESHOLD,
                 force_threshold: float = FORCE_COMPRESS_THRESHOLD):
        self.max_tokens = max_tokens
        self.warning_threshold = warning_threshold
        self.force_threshold = force_threshold
        self.alert_log: list[dict] = []
    
    def check(self, context_tokens: int, request_id: str = "") -> dict:
        """
        每次請求前呼叫，返回操作建議
        生產中：這個結果決定是否觸發壓縮
        """
        usage = context_tokens / self.max_tokens
        remaining = self.max_tokens - context_tokens
        
        if usage > 1.0:
            action = "REJECT"        # 直接拒絕請求
            level = "CRITICAL"
        elif usage > self.force_threshold:
            action = "FORCE_COMPRESS"  # 強制壓縮
            level = "ERROR"
        elif usage > self.warning_threshold:
            action = "WARN_COMPRESS"   # 建議壓縮
            level = "WARNING"
        else:
            action = "PROCEED"         # 正常
            level = "OK"
        
        result = {
            "action": action,
            "level": level,
            "usage_pct": round(usage * 100, 1),
            "context_tokens": context_tokens,
            "remaining_tokens": remaining,
            "request_id": request_id
        }
        
        if level in ["WARNING", "ERROR", "CRITICAL"]:
            self.alert_log.append(result)
        
        return result


monitor = TokenMonitor()

print("=" * 55)
print("Token 監控測試")
print("=" * 55)
print(f"{'Tokens':>10} | {'使用率':>8} | {'動作':>16} | 等級")
print("-" * 55)

test_token_counts = [5000, 30000, 60000, 80000, 105000, 130000]
icons = {"OK": "✅", "WARNING": "🟡", "ERROR": "🔴", "CRITICAL": "💥"}

for tokens in test_token_counts:
    result = monitor.check(tokens, f"req_{tokens}")
    icon = icons[result["level"]]
    print(f"{tokens:>10,} | {result['usage_pct']:>7.1f}% | {result['action']:>16} | {icon} {result['level']}")

print(f"\n告警記錄數：{len(monitor.alert_log)}")
print("\n📋 生產環境整合建議：")
print("""
  每次 LLM 呼叫前：
    result = monitor.check(context_tokens)
    if result['action'] == 'FORCE_COMPRESS':
        context = await compress_history(history)  # 觸發壓縮
    elif result['action'] == 'REJECT':
        return {"error": "Context 超限，請開新對話"}
    
    # 寫入 metrics（Grafana / Cloud Monitoring）
    metric_writer.write('context_usage_pct', result['usage_pct'])
""")

## Part 7：Context 排列順序 — 對抗 Lost-in-the-Middle

In [ ]:
# 展示：Context 排列順序對 LLM 注意力的影響
# 這是面試官喜歡追問的第二層問題

CRITICAL_INFO = "訂單 #1234 的退款金額為 NTD 2,999。"  # 關鍵資訊

# ❌ 壞的排列：關鍵資訊在中間
bad_order_context = f"""[System Prompt: 你是客服 AI...]
[User Profile: 王大明，金卡會員]

--- 對話歷史（中間區域，注意力弱） ---
{CRITICAL_INFO}  ← 關鍵資訊在這裡，但 LLM 容易忽略
其他對話...
更多對話...
更多對話...
--- 對話歷史結束 ---

[Current Query: 我的退款多少錢？]"""

# ✅ 好的排列：關鍵資訊靠近結尾（高注意力區）
good_order_context = f"""[System Prompt: 你是客服 AI...]
[User Profile: 王大明，金卡會員]

--- 對話摘要（中間，可以稍微被忽略）---
用戶報告訂單問題，已提交照片。
--- 摘要結束 ---

--- 最近對話 ---
更多對話...
--- 最近對話結束 ---

=== 關鍵資訊（重要，請優先參考）===
{CRITICAL_INFO}  ← 關鍵資訊靠近結尾
===

[Current Query: 我的退款多少錢？]
記住以上關鍵資訊，請回答用戶的問題。"""

print("=" * 55)
print("Context 排列順序對比")
print("=" * 55)

print("""
LLM 的注意力分布（長 context 時）：

注意力
  ▲
高 │██████                              ██████
   │      ████                      ████
低 │          ████████████████████████
   └──────────────────────────────────────→
      System Prompt                   Current Query
      （高注意力）                     （高注意力）
               中段對話歷史：LLM 最容易忽略！

工程應對策略：
  1. System Prompt → 放最重要的角色定義和規則
  2. User Profile  → 緊接 System Prompt
  3. 摘要 / 歷史   → 中間（可以稍微忽略）
  4. 關鍵資訊      → 靠近結尾，用明確標記框住
  5. Current Query → 最後，並在 Query 前重申關鍵指令
""")

print(f"壞排列：關鍵資訊在中間")
print(f"好排列：關鍵資訊靠近結尾 + 明確標記")

## Part 8：面試 SCOPE 框架 + 策略比較

In [ ]:
print("""
面試答題 SCOPE 框架：
─────────────────────────────────────────────────────
S → Size    先估算：這個場景的 context 多快會滿？
  「客服 Agent，每輪 300 tokens，50 輪後 = 15K tokens。
   加上 system prompt 和 RAG context，約 20K。
   Gemini Flash 有 1M，但 20K 每次推理的成本和延遲是有感的。」

C → Cost    context 長了對成本和延遲的量化影響
  「Input token 按量計費，1K tokens 約 $0.001。
   50 輪 vs 10 輪，每次呼叫成本差 4 倍。
   Attention 複雜度 O(n²)，context 翻倍、延遲接近翻兩倍。」

O → Options 你有哪些策略
  「Sliding Window：最簡單，但丟舊資訊。
   Summary Buffer：壓縮保語意，但多一次 LLM 呼叫。
   Retrieval Memory：最彈性，需要向量 DB。
   Structured State：Context 永遠固定，但需要預設 schema。」

P → Pick    選哪個？說出 trade-off
  「客服場景用 Summary Buffer + Structured State 混合：
   保留最近 10 輪完整對話（3K tokens），
   把早期對話壓縮成 case summary（500 tokens）。
   同時維護結構化 state 記錄 order_id、issue_type。」

E → Edge    你的策略在什麼情況下會失效？
  「Summary 可能丟掉關鍵的錯誤碼和具體數字。
   所以壓縮 prompt 要明確標記「以下欄位必須保留」，
   並在 structured state 同步儲存關鍵數值。
   另外加 token 監控，超過 80% 自動觸發壓縮。」
─────────────────────────────────────────────────────
""")

print("四種策略速查：")
print(f"{'策略':<20} {'複雜度':<10} {'長期記憶':<10} {'額外成本':<15} 最適場景")
print("-" * 75)
strategies = [
    ("Sliding Window",   "⭐",    "❌",    "無",              "短對話、每輪獨立"),
    ("Summary Buffer",   "⭐⭐⭐",  "△",     "LLM 壓縮呼叫",   "客服、顧問類"),
    ("Retrieval Memory", "⭐⭐⭐⭐", "✅✅",   "Vector DB",      "長期個人助理"),
    ("Structured State", "⭐⭐⭐",  "❌",    "State 提取 LLM", "Task-oriented"),
]
for s in strategies:
    print(f"  {s[0]:<20} {s[1]:<10} {s[2]:<10} {s[3]:<15} {s[4]}")